# 03 · Random Forest baseline

Hudgins time-domain features → Random Forest. Within-subject 70/15/15 split by repetition to avoid window-level leakage (windows from the same repetition can be very similar).

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
from sklearn.ensemble import RandomForestClassifier

from src.features import extract_features
from src.evaluate import per_class_report, plot_confusion_matrix, save_metrics

DATA = np.load(ROOT / 'data' / 'processed' / 'windows.npz')
windows, labels, reps, subjects = DATA['windows'], DATA['labels'], DATA['reps'], DATA['subjects']
print('windows:', windows.shape, 'classes:', np.unique(labels).size)

In [ ]:
# Within-subject split by repetition. Reps 1-6 → train, 7-8 → val, 9-10 → test.
train_mask = np.isin(reps, [1, 2, 3, 4, 5, 6])
val_mask   = np.isin(reps, [7, 8])
test_mask  = np.isin(reps, [9, 10])

X_train = extract_features(windows[train_mask])
X_val   = extract_features(windows[val_mask])
X_test  = extract_features(windows[test_mask])
y_train, y_val, y_test = labels[train_mask], labels[val_mask], labels[test_mask]
print('train/val/test:', X_train.shape, X_val.shape, X_test.shape)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
rf.fit(X_train, y_train)
val_pred = rf.predict(X_val)
test_pred = rf.predict(X_test)
print('val acc:', (val_pred == y_val).mean())
print('test acc:', (test_pred == y_test).mean())

In [ ]:
rf_metrics = per_class_report(y_test, test_pred)
save_metrics(rf_metrics, ROOT / 'results' / 'metrics' / 'rf_metrics.json')
plot_confusion_matrix(y_test, test_pred,
                      title='Random Forest — test confusion matrix',
                      save_path=ROOT / 'results' / 'figures' / 'rf_confusion.png')